# Feature engineering

### **Featue 1 - Revenue**

In [8]:
import pandas as pd

sales_df = pd.read_csv("../data/processed/valid_sales.csv")

sales_df["Revenue"] = (
    sales_df["Quantity"] *
    sales_df["Price"]
)

C:\Users\Dulini Prabhashini\AppData\Local\Temp\ipykernel_17364\3891926136.py:3: DtypeWarning: Columns (0: Invoice) have mixed types. Specify dtype option on import or set low_memory=False.
  sales_df = pd.read_csv("../data/processed/valid_sales.csv")


### **Feature 2 - Time features**

In [ ]:
sales_df["InvoiceDate"] = pd.to_datetime(
    sales_df["InvoiceDate"],
    errors="coerce"
)

sales_df["Year"] = sales_df["InvoiceDate"].dt.year
sales_df["Month"] = sales_df["InvoiceDate"].dt.month
sales_df["DayOfWeek"] = sales_df["InvoiceDate"].dt.dayofweek
sales_df["Hour"] = sales_df["InvoiceDate"].dt.hour

print(sales_df["InvoiceDate"].dtype)

print(sales_df["InvoiceDate"].isna().sum()) # output = 0 -> all InvoiceDate values were successfully converted.

datetime64[us]
0


### **Feature 3 - Weekend indicator**

In [12]:
sales_df["IsWeekend"] = (
    sales_df["DayOfWeek"] >= 5
).astype(int)

### **Feature 4 - Customer RFM**

In [13]:
reference_date = sales_df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = (
    sales_df
    .groupby("Customer ID")
    .agg(
        Recency=(
            "InvoiceDate",
            lambda x: (reference_date - x.max()).days
        ),
        Frequency=("Invoice", "nunique"),
        Monetary=("Revenue", "sum")
    )
    .reset_index()
)

### **Feature 5 - Average order value**

In [14]:
customer_orders = (
    sales_df
    .groupby("Customer ID")
    .agg(
        TotalRevenue=("Revenue", "sum"),
        TotalOrders=("Invoice", "nunique")
    )
    .reset_index()
)

customer_orders["AverageOrderValue"] = (
    customer_orders["TotalRevenue"] /
    customer_orders["TotalOrders"]
)

### **Feature 6 - Unique products purchased**

In [15]:
unique_products = (
    sales_df
    .groupby("Customer ID")["StockCode"]
    .nunique()
    .reset_index(name="UniqueProducts")
)

## Forecasting features

#### For sales forecasting we need historical lag features.

### **Feature 7 - Previous day revenue**

In [17]:
daily_sales = pd.read_csv("../data/processed/daily_sales.csv")
forecast_df = daily_sales.copy()

forecast_df["Lag_1"] = (
    forecast_df["Revenue"].shift(1)
)

### **Feature 8 - Previous 7 day revenue**

In [18]:
forecast_df["Lag_7"] = (
    forecast_df["Revenue"].shift(7)
)

### **Feature 9 - Previous 14 day revenue**

In [19]:
forecast_df["Lag_14"] = (
    forecast_df["Revenue"].shift(14)
)

### **Feature 10 - Previous 28 day revenue**

In [20]:
forecast_df["Lag_28"] = (
    forecast_df["Revenue"].shift(28)
)

### **Rolling features**

In [21]:
forecast_df["RollingMean_7"] = (
    forecast_df["Revenue"]
    .shift(1)
    .rolling(7)
    .mean()
)

forecast_df["RollingMean_14"] = (
    forecast_df["Revenue"]
    .shift(1)
    .rolling(14)
    .mean()
)

forecast_df["RollingMean_28"] = (
    forecast_df["Revenue"]
    .shift(1)
    .rolling(28)
    .mean()
)